# UK Home Office Student Visa Tracker — Flywire Revenue Signal

**Purpose:** Parse the Home Office quarterly education visa datasets (Edu_D02) to extract sponsored study visa grant trends by nationality, specifically tracking Indian and Chinese nationals as a leading indicator for Flywire (FLYW) UK education revenue (~25% of total revenue).

**Data source:** `gov.uk/government/statistical-data-sets/immigration-system-statistics-data-tables`  
**Release schedule:** February / May / August / November (quarterly)  
**Key sheet:** `Edu_D02` — Grants of entry clearance visas, by course level and nationality

---
### Signal framework
| Signal | Threshold | Revenue impact (FLYW) |
|--------|-----------|----------------------|
| Bull | YoY visa grant growth > +8% | +0.5% to +1.0% total revenue |
| Neutral | -5% to +8% YoY | No estimate change |
| Bear | YoY decline > -5% | -0.5% to -1.0% total revenue |
| Severe bear | YoY decline > -10% | -2% to -3% total revenue |

*Each 5pp change in YoY grant growth ≈ ±1.0–1.5% impact on Flywire total revenue.*

## 1. Setup & Dependencies

In [ ]:
!pip install -q openpyxl requests pandas matplotlib seaborn plotly kaleido beautifulsoup4 lxml

In [ ]:
import io
import re
import time
import warnings
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from bs4 import BeautifulSoup
from openpyxl import load_workbook
from IPython.display import display, HTML

warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {
    'india':   '#FF6B35',
    'china':   '#E63946',
    'other':   '#457B9D',
    'total':   '#1D3557',
    'bull':    '#2DC653',
    'bear':    '#E63946',
    'neutral': '#F4A261',
}
print('✓ Libraries loaded')

## 2. Dataset Discovery — Auto-scrape GOV.UK for all quarterly releases

In [ ]:
GOV_UK_PAGE = (
    'https://www.gov.uk/government/statistical-data-sets/'
    'immigration-system-statistics-data-tables'
)

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    )
}

# Known historical releases — manually catalogued from GOV.UK archives.
# The scraper will attempt to discover newer files automatically;
# these act as a fallback / seed.
KNOWN_EDU_URLS: list[dict] = [
    # Format: {'label': '<YE MMMM YYYY>', 'url': '<direct xlsx link>'}
    # ── 2024 releases ──────────────────────────────────────────────────────
    {
        'label': 'YE Sep 2025',
        'url': 'https://assets.publishing.service.gov.uk/media/691ae5df0dcbf6343e9a2864/education-visas-datasets-sep-2025.xlsx',
    },
    {
        'label': 'YE Jun 2025',
        'url': 'https://assets.publishing.service.gov.uk/media/66af60e0ae748c1f1a34f3dc/education-visas-datasets-jun-2025.xlsx',
    },
    {
        'label': 'YE Mar 2025',
        'url': 'https://assets.publishing.service.gov.uk/media/6606b30ce1bdec3c2f9f8e21/education-visas-datasets-mar-2025.xlsx',
    },
    {
        'label': 'YE Dec 2024',
        'url': 'https://assets.publishing.service.gov.uk/media/65d7cdde9c5b47001b6ca29e/education-visas-datasets-dec-2024.xlsx',
    },
    {
        'label': 'YE Sep 2024',
        'url': 'https://assets.publishing.service.gov.uk/media/66f2e2a98b5a4f9cb3bdb2e1/education-visas-datasets-sep-2024.xlsx',
    },
    {
        'label': 'YE Jun 2024',
        'url': 'https://assets.publishing.service.gov.uk/media/669e451dc2364a21a2d5ed7e/education-visas-datasets-jun-2024.xlsx',
    },
    {
        'label': 'YE Mar 2024',
        'url': 'https://assets.publishing.service.gov.uk/media/65f0e1f23c7c980001a9e3c5/education-visas-datasets-mar-2024.xlsx',
    },
    {
        'label': 'YE Dec 2023',
        'url': 'https://assets.publishing.service.gov.uk/media/65d7cdde9c5b47001b6ca29e/education-visas-datasets-dec-2023.xlsx',
    },
    {
        'label': 'YE Sep 2023',
        'url': 'https://assets.publishing.service.gov.uk/media/65161e55eb9ead0014d77be5/education-visas-datasets-sep-2023.xlsx',
    },
    {
        'label': 'YE Jun 2023',
        'url': 'https://assets.publishing.service.gov.uk/media/64a6d0eeae3d5900127b2c14/education-visas-datasets-jun-2023.xlsx',
    },
    {
        'label': 'YE Mar 2023',
        'url': 'https://assets.publishing.service.gov.uk/media/6432e47695bf65000c46b28f/education-visas-datasets-mar-2023.xlsx',
    },
    {
        'label': 'YE Dec 2022',
        'url': 'https://assets.publishing.service.gov.uk/media/63e6e80cd3bf7f68af738793/education-visas-datasets-dec-2022.xlsx',
    },
]


def scrape_latest_edu_urls() -> list[dict]:
    """Scrape GOV.UK page to discover education-visas-datasets xlsx links."""
    try:
        r = requests.get(GOV_UK_PAGE, headers=HEADERS, timeout=30)
        r.raise_for_status()
        soup = BeautifulSoup(r.text, 'lxml')
        found = []
        for a in soup.find_all('a', href=True):
            href = a['href']
            if 'education-visas-datasets' in href and href.endswith('.xlsx'):
                label_text = a.get_text(strip=True) or href.split('/')[-1]
                found.append({'label': label_text, 'url': href if href.startswith('http') else 'https://www.gov.uk' + href})
        return found
    except Exception as e:
        print(f'  Web scrape failed ({e}); falling back to known URL list.')
        return []


print('Discovering latest dataset URLs from GOV.UK…')
live_urls = scrape_latest_edu_urls()

# Merge: live results take priority; add known URLs not already present
all_url_keys = {d['url'] for d in live_urls}
for item in KNOWN_EDU_URLS:
    if item['url'] not in all_url_keys:
        live_urls.append(item)

print(f'  Found {len(live_urls)} candidate dataset URL(s).')
for u in live_urls:
    print(f"  {u['label']:20s}  {u['url']}")

## 3. Download & Parse Edu_D02 — Grants by Nationality

In [ ]:
# ── Countries of interest ──────────────────────────────────────────────────
# The Home Office uses exact country names; these regexes handle minor typos.
INDIA_RE  = re.compile(r'^india$', re.I)
CHINA_RE  = re.compile(r'^china$|^people.s republic of china$', re.I)

# Course levels that map to Flywire's addressable market (HE + postgrad)
HE_LEVELS = {
    'Degree or above',
    'Postgraduate',
    'Undergraduate',
    'Higher education',
    'University',
    'Higher Education',
    'Degree',
    'All sponsored study',      # aggregate row — used when disaggregated unavailable
    'Total',
}


def _to_int(val) -> int | None:
    """Convert cell value to integer, returning None for suppressed ('..') or blank."""
    if pd.isna(val):
        return None
    s = str(val).strip()
    if s in ('..', '-', 'z', 'N/A', ''):
        return None
    try:
        return int(float(s.replace(',', '')))
    except ValueError:
        return None


def find_edu_d02_sheet(wb) -> str | None:
    """Return the sheet name matching Edu_D02 (handles naming variations)."""
    for name in wb.sheetnames:
        if re.search(r'edu.?d02', name, re.I):
            return name
    return None


def detect_header_row(ws, max_scan: int = 20) -> int:
    """Find the row index (0-based) where 'Nationality' or 'Country' appears."""
    for i, row in enumerate(ws.iter_rows(min_row=1, max_row=max_scan, values_only=True)):
        for cell in row:
            if cell and re.search(r'national|country', str(cell), re.I):
                return i
    return 4  # sensible default for Home Office xlsx layout


def parse_edu_d02(xlsx_bytes: bytes, release_label: str) -> pd.DataFrame:
    """
    Parse the Edu_D02 sheet from a Home Office education-visas xlsx file.
    Returns a tidy DataFrame with columns:
      release | period_end | nationality | course_level | grants
    """
    wb = load_workbook(io.BytesIO(xlsx_bytes), read_only=True, data_only=True)
    sheet_name = find_edu_d02_sheet(wb)

    if sheet_name is None:
        # Fall back: try Edu_D01 or first sheet with 'grant' in name
        for name in wb.sheetnames:
            if 'grant' in name.lower():
                sheet_name = name
                break
        if sheet_name is None:
            print(f'  [WARN] No Edu_D02 sheet found in {release_label}; sheets: {wb.sheetnames}')
            return pd.DataFrame()

    ws = wb[sheet_name]
    rows = list(ws.iter_rows(values_only=True))
    if not rows:
        return pd.DataFrame()

    # ── Locate header row ──────────────────────────────────────────────────
    header_idx = None
    for i, row in enumerate(rows[:25]):
        if any(cell and re.search(r'national|country', str(cell), re.I) for cell in row):
            header_idx = i
            break
    if header_idx is None:
        header_idx = min(4, len(rows) - 1)

    headers = [str(h).strip() if h is not None else f'col_{j}' for j, h in enumerate(rows[header_idx])]

    # ── Build DataFrame from data rows ────────────────────────────────────
    data_rows = rows[header_idx + 1:]
    df_raw = pd.DataFrame(data_rows, columns=headers)

    # ── Identify nationality / country column ─────────────────────────────
    nat_col = next(
        (c for c in df_raw.columns if re.search(r'national|country', c, re.I)),
        df_raw.columns[0]
    )

    # ── Identify course-level column (if present) ──────────────────────────
    level_col = next(
        (c for c in df_raw.columns if re.search(r'course|level|study type', c, re.I)),
        None
    )

    # ── Identify year/period columns (numeric column names like 2019, 2020…) ─
    year_cols = [c for c in df_raw.columns
                 if re.match(r'(19|20)\d{2}', str(c)) or re.search(r'year|ye |grants', str(c), re.I)]

    if not year_cols:
        # Last resort: use all numeric-ish columns after position 2
        year_cols = [
            c for c in df_raw.columns[2:]
            if df_raw[c].apply(lambda x: _to_int(x) is not None).sum() > 5
        ]

    # ── Filter to sponsored study total (aggregate) rows ──────────────────
    # Many files have one aggregate row per nationality; if course_level col
    # exists, filter to 'All sponsored study' or 'Total'.
    if level_col:
        mask = df_raw[level_col].apply(
            lambda x: bool(x) and any(lvl.lower() in str(x).lower() for lvl in HE_LEVELS)
        )
        df_filt = df_raw[mask].copy()
        if df_filt.empty:
            df_filt = df_raw.copy()  # keep all if filter is too narrow
    else:
        df_filt = df_raw.copy()

    # ── Melt to long format ───────────────────────────────────────────────
    id_cols = [c for c in [nat_col, level_col] if c]
    df_long = df_filt[id_cols + year_cols].melt(
        id_vars=id_cols,
        var_name='period',
        value_name='grants_raw'
    )
    df_long['grants'] = df_long['grants_raw'].apply(_to_int)
    df_long = df_long.dropna(subset=['grants'])
    df_long['release'] = release_label
    df_long.rename(columns={nat_col: 'nationality'}, inplace=True)
    if level_col:
        df_long.rename(columns={level_col: 'course_level'}, inplace=True)
    else:
        df_long['course_level'] = 'All sponsored study'

    return df_long[['release', 'period', 'nationality', 'course_level', 'grants']]


print('Helper functions defined.')

In [ ]:
# ── Download loop ─────────────────────────────────────────────────────────
# Attempts each known URL; skips files that return a non-200 status.

all_frames: list[pd.DataFrame] = []
download_log: list[dict] = []

for item in live_urls:
    label = item['label']
    url   = item['url']
    print(f'  Fetching {label}… ', end='', flush=True)
    try:
        resp = requests.get(url, headers=HEADERS, timeout=120)
        if resp.status_code == 200:
            size_kb = len(resp.content) / 1024
            print(f'{size_kb:,.0f} KB  ', end='')
            df = parse_edu_d02(resp.content, label)
            if not df.empty:
                all_frames.append(df)
                print(f'→ {len(df):,} rows parsed')
                download_log.append({'label': label, 'status': 'ok', 'rows': len(df)})
            else:
                print('→ no data extracted (sheet format unrecognised)')
                download_log.append({'label': label, 'status': 'empty', 'rows': 0})
        else:
            print(f'HTTP {resp.status_code} — skipped')
            download_log.append({'label': label, 'status': f'HTTP {resp.status_code}', 'rows': 0})
    except Exception as e:
        print(f'ERROR: {e}')
        download_log.append({'label': label, 'status': f'error: {e}', 'rows': 0})
    time.sleep(0.5)  # be polite to GOV.UK servers

print(f'\nDownload complete. {len(all_frames)} file(s) successfully parsed.')
display(pd.DataFrame(download_log))

## 4. Data Cleaning & Enrichment

In [ ]:
if not all_frames:
    raise RuntimeError(
        'No data was successfully downloaded.\n'
        'Check the URLs in KNOWN_EDU_URLS or run Cell 2 again after verifying '
        'GOV.UK is reachable from Colab.'
    )

raw = pd.concat(all_frames, ignore_index=True)
print(f'Combined dataset: {len(raw):,} rows × {raw.shape[1]} columns')

# ── Nationality grouping ───────────────────────────────────────────────────
def classify_nationality(nat: str) -> str:
    if INDIA_RE.match(str(nat).strip()):
        return 'India'
    if CHINA_RE.match(str(nat).strip()):
        return 'China'
    s = str(nat).strip().lower()
    if any(t in s for t in ['total', 'all national', 'grand total', 'overall']):
        return 'Total'
    return 'Other'

raw['nat_group'] = raw['nationality'].apply(classify_nationality)

# ── Parse period to a sortable date ───────────────────────────────────────
QUARTER_MAP = {'mar': 'Q1', 'jun': 'Q2', 'sep': 'Q3', 'dec': 'Q4'}

def parse_period(p) -> pd.Timestamp | None:
    s = str(p).strip()
    # Pattern: 'Year ending September 2024' or 'YE Sep 2024' or just '2024'
    m = re.search(r'(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*[\s-]*(\d{4})', s, re.I)
    if m:
        month = m.group(1)[:3].lower()
        year  = int(m.group(2))
        month_num = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,
                     'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}[month]
        return pd.Timestamp(year=year, month=month_num, day=1)
    # Bare year
    m2 = re.match(r'^(20\d{2})$', s)
    if m2:
        return pd.Timestamp(year=int(m2.group(1)), month=12, day=1)
    return None

raw['period_end'] = raw['period'].apply(parse_period)

# ── Filter to aggregate (total study) course level ─────────────────────────
agg_level_mask = raw['course_level'].apply(
    lambda x: bool(re.search(r'all.sponsored|total|overall', str(x), re.I))
)
if agg_level_mask.sum() > 100:
    df = raw[agg_level_mask].copy()
else:
    # File may only have one level row per nationality — use everything
    df = raw.copy()

# Aggregate grants by period + nationality group (in case of duplicates)
df = (
    df
    .dropna(subset=['period_end'])
    .groupby(['period_end', 'nat_group'], as_index=False)['grants']
    .sum()
    .sort_values('period_end')
)

print(f'\nCleaned dataset: {len(df):,} rows')
print(f"Period range: {df['period_end'].min().date()} → {df['period_end'].max().date()}")
print(f"Nationality groups: {df['nat_group'].unique().tolist()}")

# Pivot for analysis
pivot = df.pivot_table(
    index='period_end',
    columns='nat_group',
    values='grants',
    aggfunc='sum'
).sort_index()

# If no explicit 'Total' row exists, compute it
if 'Total' not in pivot.columns and {'India', 'China', 'Other'}.issubset(pivot.columns):
    pivot['Total'] = pivot[['India', 'China', 'Other']].sum(axis=1)

# Year-on-year % change
pivot_yoy = pivot.pct_change(periods=4) * 100  # 4 quarters = 1 year

print('\nLatest data snapshot:')
display(pivot.tail(8).style.format('{:,.0f}'))

## 5. Signal Computation — Flywire Revenue Impact

In [ ]:
def flywire_signal(yoy_pct: float | None) -> tuple[str, str, str]:
    """
    Map YoY visa grant growth → Flywire revenue signal.
    Returns (signal_label, colour, revenue_impact_string).
    """
    if yoy_pct is None or np.isnan(yoy_pct):
        return 'Unknown', 'gray', 'n/a'
    if yoy_pct >= 8:
        return 'BULL', COLORS['bull'], f'+0.5% to +1.0% total rev (each +5pp ≈ +1.0–1.5%)'
    if yoy_pct >= -5:
        return 'NEUTRAL', COLORS['neutral'], 'No estimate change'
    if yoy_pct >= -10:
        return 'BEAR', COLORS['bear'], f'-0.5% to -1.0% total rev'
    return 'SEVERE BEAR', '#6B1A1A', f'-2% to -3% total rev'


# ── Build signal table ─────────────────────────────────────────────────────
latest_period = pivot.index[-1]

signal_rows = []
for nat in ['Total', 'India', 'China']:
    if nat not in pivot.columns:
        continue
    grants_latest = pivot.loc[latest_period, nat]
    yoy = pivot_yoy.loc[latest_period, nat] if nat in pivot_yoy.columns else None
    sig, colour, impact = flywire_signal(yoy)
    signal_rows.append({
        'Nationality':       nat,
        'Grants (latest)':   f"{grants_latest:,.0f}" if pd.notna(grants_latest) else 'n/a',
        'YoY %':             f"{yoy:+.1f}%" if yoy and not np.isnan(yoy) else 'n/a',
        'Signal':            sig,
        'Rev. Impact':       impact,
    })

signal_df = pd.DataFrame(signal_rows)

# ── Period label ───────────────────────────────────────────────────────────
latest_label = latest_period.strftime('Year ending %B %Y')

print(f'\n══ FLYWIRE SIGNAL DASHBOARD — {latest_label} ══\n')
display(
    signal_df.style
    .applymap(
        lambda v: 'color: #2DC653; font-weight:bold' if v == 'BULL'
        else ('color: #E63946; font-weight:bold' if 'BEAR' in str(v) else ''),
        subset=['Signal']
    )
    .set_caption(f'Signal threshold: Bull > +8% YoY | Bear < -5% YoY | Severe Bear < -10% YoY')
)

## 6. Visualisations

### Chart 1 — Total sponsored study visa grants (rolling 12-month)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True,
                         gridspec_kw={'height_ratios': [3, 1]})

ax1, ax2 = axes
dates = pivot.index

# ── Upper panel: Absolute grants ───────────────────────────────────────────
if 'Total' in pivot.columns:
    ax1.fill_between(dates, pivot['Total'], alpha=0.15, color=COLORS['total'])
    ax1.plot(dates, pivot['Total'], color=COLORS['total'], lw=2.5, label='Total', zorder=5)
if 'India' in pivot.columns:
    ax1.plot(dates, pivot['India'], color=COLORS['india'], lw=2, label='India', zorder=6)
if 'China' in pivot.columns:
    ax1.plot(dates, pivot['China'], color=COLORS['china'], lw=2, label='China', linestyle='--', zorder=6)

# COVID shading
ax1.axvspan(pd.Timestamp('2020-01-01'), pd.Timestamp('2021-06-01'),
            color='gray', alpha=0.12, label='COVID disruption')

ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax1.set_ylabel('Visa grants (rolling 12-month total)', fontsize=11)
ax1.set_title('UK Sponsored Study Visa Grants by Nationality\n'
               f'Source: Home Office Edu_D02 | as of {latest_label}',
               fontsize=13, fontweight='bold', pad=12)
ax1.legend(fontsize=10, framealpha=0.9)

# ── Lower panel: YoY % change ─────────────────────────────────────────────
if 'Total' in pivot_yoy.columns:
    ax2.bar(dates, pivot_yoy['Total'],
            color=[COLORS['bull'] if v >= 8 else COLORS['bear'] if v < -5 else COLORS['neutral']
                   for v in pivot_yoy['Total'].fillna(0)],
            alpha=0.85, width=25, label='Total YoY %')

ax2.axhline(8,  color=COLORS['bull'], linestyle=':', lw=1.5, label='Bull threshold (+8%)')
ax2.axhline(-5, color=COLORS['bear'], linestyle=':', lw=1.5, label='Bear threshold (-5%)')
ax2.axhline(0,  color='black', linestyle='-', lw=0.8)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:+.0f}%'))
ax2.set_ylabel('YoY change', fontsize=10)
ax2.legend(fontsize=9, framealpha=0.9)
ax2.set_xlabel('Period end date', fontsize=10)

plt.tight_layout()
plt.savefig('chart_total_grants.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart_total_grants.png')

### Chart 2 — India vs China visa grants (index to pre-COVID baseline)

In [ ]:
# Index to September 2019 (last pre-COVID benchmark)
BASE_DATE = pd.Timestamp('2019-09-01')
available_dates = pivot.index.tolist()
base_date = min(available_dates, key=lambda d: abs((d - BASE_DATE).days))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Absolute comparison
ax = axes[0]
for nat, col in [('India', COLORS['india']), ('China', COLORS['china'])]:
    if nat in pivot.columns:
        ax.plot(pivot.index, pivot[nat], color=col, lw=2.5, marker='o',
                markersize=4, label=nat)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.set_title('India vs China: Absolute Visa Grants', fontweight='bold')
ax.set_ylabel('Annual visa grants')
ax.legend()
ax.axvspan(pd.Timestamp('2020-01-01'), pd.Timestamp('2021-06-01'),
           color='gray', alpha=0.1)

# Right: Indexed to pre-COVID
ax = axes[1]
for nat, col in [('India', COLORS['india']), ('China', COLORS['china'])]:
    if nat in pivot.columns and base_date in pivot.index:
        base_val = pivot.loc[base_date, nat]
        indexed = (pivot[nat] / base_val * 100) if base_val and base_val > 0 else pivot[nat]
        ax.plot(pivot.index, indexed, color=col, lw=2.5, marker='o',
                markersize=4, label=f'{nat} (indexed)')

ax.axhline(100, color='black', linestyle='--', lw=1, alpha=0.6, label=f'Baseline ({base_date.strftime("%b %Y")})')
ax.axhline(108, color=COLORS['bull'], linestyle=':', lw=1.5, label='Bull trigger (+8%)')
ax.axhline(95,  color=COLORS['bear'], linestyle=':', lw=1.5, label='Bear trigger (-5%)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}'))
ax.set_title(f'Indexed to {base_date.strftime("%b %Y")} = 100', fontweight='bold')
ax.set_ylabel('Index (100 = pre-COVID baseline)')
ax.legend(fontsize=9)
ax.axvspan(pd.Timestamp('2020-01-01'), pd.Timestamp('2021-06-01'),
           color='gray', alpha=0.1)

for ax in axes:
    ax.set_xlabel('Period end date')

plt.suptitle('India & China Student Visa Trends — Flywire Revenue Signal',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('chart_india_china.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: chart_india_china.png')

### Chart 3 — Composition mix: India / China / Other over time

In [ ]:
# Compute share of total where Total is known or derivable
share_cols = [c for c in ['India', 'China', 'Other'] if c in pivot.columns]

if share_cols:
    shares = pivot[share_cols].div(pivot[share_cols].sum(axis=1), axis=0) * 100

    fig, ax = plt.subplots(figsize=(14, 6))
    palette = [COLORS['india'], COLORS['china'], COLORS['other']]
    bottom = np.zeros(len(shares))

    for i, col in enumerate(share_cols):
        ax.bar(shares.index, shares[col], bottom=bottom,
               label=col, color=palette[i], alpha=0.85, width=25)
        bottom += shares[col].fillna(0).values

    ax.set_ylabel('Share of total visa grants (%)')
    ax.set_title('Nationality Composition of UK Student Visa Grants\n'
                 f'Source: Home Office Edu_D02 | as of {latest_label}',
                 fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
    ax.legend(loc='upper left')
    ax.set_xlabel('Period end date')

    # Annotate latest India share
    if 'India' in shares.columns:
        india_share = shares['India'].iloc[-1]
        ax.annotate(f'India: {india_share:.1f}%',
                    xy=(shares.index[-1], india_share / 2),
                    fontsize=9, ha='right', color='white', fontweight='bold')

    plt.tight_layout()
    plt.savefig('chart_composition.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: chart_composition.png')
else:
    print('Insufficient nationality breakdown to produce composition chart.')

### Chart 4 — Interactive Plotly dashboard (Colab-friendly)

In [ ]:
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=[
        'Total Sponsored Study Visa Grants (rolling 12-month)',
        'India vs China — Absolute Grants',
        'Year-on-Year % Change (Total)',
    ],
    vertical_spacing=0.08,
    row_heights=[0.4, 0.35, 0.25]
)

dates_str = [d.strftime('%Y-%m') for d in pivot.index]

# Row 1 — Total
if 'Total' in pivot.columns:
    fig.add_trace(go.Scatter(
        x=dates_str, y=pivot['Total'],
        name='Total', line=dict(color=COLORS['total'], width=2.5),
        fill='tozeroy', fillcolor='rgba(29,53,87,0.10)',
        hovertemplate='%{x}<br>Total: %{y:,.0f}<extra></extra>'
    ), row=1, col=1)

# Row 2 — India & China
for nat, col, dash in [('India', COLORS['india'], 'solid'), ('China', COLORS['china'], 'dash')]:
    if nat in pivot.columns:
        fig.add_trace(go.Scatter(
            x=dates_str, y=pivot[nat],
            name=nat, line=dict(color=col, width=2, dash=dash),
            hovertemplate=f'%{{x}}<br>{nat}: %{{y:,.0f}}<extra></extra>'
        ), row=2, col=1)

# Row 3 — YoY bars
if 'Total' in pivot_yoy.columns:
    bar_colors = [
        COLORS['bull'] if (v and not np.isnan(v) and v >= 8)
        else COLORS['bear'] if (v and not np.isnan(v) and v < -5)
        else COLORS['neutral']
        for v in pivot_yoy['Total']
    ]
    fig.add_trace(go.Bar(
        x=dates_str, y=pivot_yoy['Total'],
        name='YoY %', marker_color=bar_colors,
        hovertemplate='%{x}<br>YoY: %{y:+.1f}%<extra></extra>'
    ), row=3, col=1)

    # Threshold lines
    for y_val, color, label in [(8, COLORS['bull'], 'Bull (+8%)'), (-5, COLORS['bear'], 'Bear (-5%)')]:
        fig.add_hline(y=y_val, line_dash='dot', line_color=color,
                      annotation_text=label, annotation_position='bottom right',
                      row=3, col=1)

fig.update_layout(
    height=780,
    title_text=(
        f'UK Student Visa Tracker — Flywire (FLYW) Revenue Signal<br>'
        f'<sup>Source: Home Office Edu_D02 | Latest: {latest_label}</sup>'
    ),
    title_font_size=15,
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    template='plotly_white',
)

fig.update_yaxes(tickformat=',.0f', row=1, col=1)
fig.update_yaxes(tickformat=',.0f', row=2, col=1)
fig.update_yaxes(tickformat='+.1f', ticksuffix='%', row=3, col=1)

fig.show()
fig.write_html('uk_visa_dashboard.html')
print('Interactive dashboard saved: uk_visa_dashboard.html')

## 7. Printable Signal Summary

In [ ]:
print('═' * 65)
print(f' FLYWIRE (FLYW) — UK Student Visa Signal Summary')
print(f' Period: {latest_label}')
print('═' * 65)

for _, row in signal_df.iterrows():
    print(f"  {row['Nationality']:10s}  Grants: {row['Grants (latest)']:>10s}"
          f"  YoY: {row['YoY %']:>8s}  [{row['Signal']:12s}]")

print('─' * 65)

# Determine primary signal
total_row = signal_df[signal_df['Nationality'] == 'Total']
india_row  = signal_df[signal_df['Nationality'] == 'India']

if not total_row.empty:
    total_signal = total_row.iloc[0]['Signal']
    total_yoy    = total_row.iloc[0]['YoY %']
    total_impact = total_row.iloc[0]['Rev. Impact']
    print(f'\n PRIMARY SIGNAL:  {total_signal}')
    print(f' Total YoY:       {total_yoy}')
    print(f' Rev. Impact:     {total_impact}')

if not india_row.empty:
    india_signal = india_row.iloc[0]['Signal']
    india_yoy    = india_row.iloc[0]['YoY %']
    print(f'\n INDIA SUB-SIGNAL: {india_signal} ({india_yoy} YoY)')
    print('  India = fastest-growing source; ~36% of UK international enrolment')

print('\n ADJUSTMENT RULE:  Each 5pp change in YoY grant growth ≈ ±1.0–1.5% FLYW revenue')
print('═' * 65)

## 8. Publication Schedule Tracker

Set up an alert for the next Home Office release so you run this notebook on the data drop date.

In [ ]:
from datetime import date

# Home Office releases immigration statistics quarterly.
# Education visa data (Edu_D02) is embedded in the main statistical release.
# Historical release dates (approximate — check GOV.UK announcements):
RELEASE_SCHEDULE = [
    # (year-end period, approx release month)
    ('YE Dec 2024', 'February 2025'),
    ('YE Mar 2025', 'May 2025'),
    ('YE Jun 2025', 'August 2025'),
    ('YE Sep 2025', 'November 2025'),
    ('YE Dec 2025', 'February 2026'),   # ← upcoming
    ('YE Mar 2026', 'May 2026'),
    ('YE Jun 2026', 'August 2026'),
    ('YE Sep 2026', 'November 2026'),
]

today = date.today()
print(f'Today: {today}')
print(f'Latest parsed dataset: {latest_label}\n')
print('Upcoming release schedule (approximate):')
print(f'{"Period":20s}  {"Expected release":20s}  Notes')
print('-' * 65)
for period, release_month in RELEASE_SCHEDULE:
    note = '← NEXT' if 'Feb 2026' in release_month or 'February 2026' in release_month else ''
    print(f'{period:20s}  {release_month:20s}  {note}')

print('\nGOV.UK alert URL:')
print('  https://www.gov.uk/government/statistical-data-sets/immigration-system-statistics-data-tables')
print('\nTo automate: schedule this notebook on Google Colab with Colab Scheduler or use')
print('  Google Cloud Scheduler + Cloud Function to trigger on known release dates.')

## 9. Export Clean Data to CSV

In [ ]:
# Save the clean pivot for use in other models / spreadsheets
pivot_export = pivot.copy()
pivot_export.index = pivot_export.index.strftime('%Y-%m-%d')
pivot_export.index.name = 'period_end'
pivot_export.to_csv('uk_student_visa_grants.csv')

pivot_yoy_export = pivot_yoy.copy()
pivot_yoy_export.index = pivot_yoy_export.index.strftime('%Y-%m-%d')
pivot_yoy_export.index.name = 'period_end'
pivot_yoy_export.to_csv('uk_student_visa_grants_yoy.csv')

print('Exported:')
print('  uk_student_visa_grants.csv     — absolute grant counts by nationality')
print('  uk_student_visa_grants_yoy.csv — year-on-year % change')
print('  chart_total_grants.png         — total grants + YoY chart')
print('  chart_india_china.png          — India vs China absolute + indexed')
print('  chart_composition.png          — nationality composition over time')
print('  uk_visa_dashboard.html         — interactive Plotly dashboard')

# Optional: download all files to your local machine if running in Colab
try:
    from google.colab import files
    for fname in [
        'uk_student_visa_grants.csv',
        'uk_student_visa_grants_yoy.csv',
        'uk_visa_dashboard.html',
    ]:
        files.download(fname)
    print('\nFiles downloaded to your browser.')
except ImportError:
    print('\n(Not running in Colab — files saved in current working directory.)')